In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

In [4]:
# Data import and cleaning from previous milestone
base_url = "https://huggingface.co/datasets/google-research-datasets/go_emotions/resolve/main/simplified/"

train_df = pd.read_parquet(base_url + "train-00000-of-00001.parquet")
val_df = pd.read_parquet(base_url + "validation-00000-of-00001.parquet")
test_df = pd.read_parquet(base_url + "test-00000-of-00001.parquet")

# Label and group mappings 
LABEL_NAMES = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]
# Map each label to its corresponding group
GROUP_MAP = {
    "admiration":"positive","amusement":"positive","approval":"positive",
    "caring":"positive","desire":"positive","excitement":"positive",
    "gratitude":"positive","joy":"positive","love":"positive",
    "optimism":"positive","pride":"positive","relief":"positive",
    "anger":"negative","annoyance":"negative","disappointment":"negative",
    "disapproval":"negative","disgust":"negative","embarrassment":"negative",
    "fear":"negative","grief":"negative","nervousness":"negative",
    "remorse":"negative","sadness":"negative",
    "confusion":"ambiguous","curiosity":"ambiguous",
    "realization":"ambiguous","surprise":"ambiguous",
    "neutral":"neutral"
}
# Function to convert label IDs to their corresponding groups
def labels_to_groups_all(label_ids):
    names = [LABEL_NAMES[i] for i in label_ids]
    groups = [GROUP_MAP[n] for n in names]
    return list(dict.fromkeys(groups))

# Process each dataset split
def process_split(df):
    df = df.copy()
    df["all_groups"] = df["labels"].apply(labels_to_groups_all)
    df["num_distinct_groups"] = df["all_groups"].apply(len)
    clean = df[df["num_distinct_groups"] == 1].copy()
    clean["group_label"] = clean["all_groups"].apply(lambda g: g[0])
    conflict = df[df["num_distinct_groups"] > 1].copy()
    return clean, conflict

# Process all dataset splits
train_clean, train_conflict = process_split(train_df)
val_clean, val_conflict = process_split(val_df)
test_clean, test_conflict = process_split(test_df)

def clean_text(text):
    text = re.sub(r"\[NAME\]|\[RELIGION\]", "", text) #remove hidden placeholder words
    text = re.sub(r"http\S+", "", text)               #remove links
    text = re.sub(r"\s+", " ", text).strip()          #remove extra whitespace   
    return text

train_clean["clean_text"] = train_clean["text"].apply(clean_text)
val_clean["clean_text"] = val_clean["text"].apply(clean_text)
test_clean["clean_text"] = test_clean["text"].apply(clean_text)

train_clean = train_clean[train_clean["clean_text"].str.strip() != ""]
val_clean = val_clean[val_clean["clean_text"].str.strip() != ""]
test_clean = test_clean[test_clean["clean_text"].str.strip() != ""]

In [ ]:
# Making a naive majority class predition benchmark
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

naive_majority_classifier = DummyClassifier(strategy="most_frequent")
naive_majority_classifier.fit(train_clean["clean_text"], train_clean["group_label"])
# The naive majority classifier should just guess "positive" as the group label for every prediction

predictions = naive_majority_classifier.predict(test_clean["clean_text"])

accuracy = accuracy_score(test_clean["group_label"], predictions)

print(f"Predictions: {predictions}\nPrediction Accuracy: {accuracy}")
# The accuracy should be 0.3705987666600358

Predictions: ['positive' 'positive' 'positive' ... 'positive' 'positive' 'positive']
Prediction Accuracy: 0.3705987666600358
